## 1. Setup & Imports

In [ ]:
import os
os.environ['MPLBACKEND'] = 'tkagg'  

import warnings
warnings.filterwarnings('ignore')

import matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit
from datetime import timedelta

plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.family': 'DejaVu Sans',
    'axes.grid': True,
    'grid.alpha': 0.3
})

SEED = 42
np.random.seed(SEED)

print('✅ Libraries loaded successfully')

✅ Libraries loaded successfully


## 2. Load & Preprocess Dataset

In [2]:
# ── Load pre-processed earthquake dataset ──────────────────────────────────
df_raw = pd.read_csv('final_preprocessed_earthquake_data.csv', low_memory=False)
print(f'Raw shape: {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')

Raw shape: (175947, 47)
Columns: ['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'nst', 'gap', 'rms', 'net', 'id', 'updated', 'place', 'type', 'status', 'year', 'month', 'day', 'hour', 'day_of_week', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'dist_to_cluster_center', 'hours_since_last_event', 'prev_mag', 'rolling_mag_5', 'depth_log', 'gap_transformed', 'rms_transformed', 'anomaly_score', 'is_anomaly', 'pca_1', 'pca_2', 'pca_3', 'dbscan_cluster', 'lat_bin', 'lon_bin', 'grid_id', 'spatial_lag_mag', 'spatial_lag_depth', 'local_heterogeneity_mag', 'moran_quadrant', 'day_of_year', 'season', 'lisa_cluster']


In [3]:
# ── Select and clean core temporal columns ─────────────────────────────────
core_cols = ['time', 'latitude', 'longitude', 'depth', 'mag', 'magType',
             'type', 'place', 'grid_id', 'lat_bin', 'lon_bin']

# Only keep columns that exist
core_cols = [c for c in core_cols if c in df_raw.columns]
df = df_raw[core_cols].copy()

# Parse time
df['time'] = pd.to_datetime(df['time'], utc=True, errors='coerce')
df = df.dropna(subset=['time', 'mag', 'latitude', 'longitude'])

# Filter earthquakes only
if 'type' in df.columns:
    df = df[df['type'] == 'earthquake'].copy()

# Remove duplicates and sort
df = df.drop_duplicates(subset=['time', 'latitude', 'longitude'])
df = df.sort_values('time').reset_index(drop=True)

print(f'Clean shape: {df.shape}')
print(f'Date range: {df["time"].min()} → {df["time"].max()}')
print(f'Magnitude range: {df["mag"].min():.1f} – {df["mag"].max():.1f}')
df[['time','latitude','longitude','depth','mag']].describe()

Clean shape: (3153, 11)
Date range: 2000-01-01 11:22:57+00:00 → 2025-05-26 22:14:17+00:00
Magnitude range: 4.5 – 7.3


,latitude,longitude,depth,mag
count,3153.000000,3153.000000,3153.000000,3153.00000
mean,-2.925395,-8.441238,54.971366,4.81125
std,31.601950,109.342550,83.874967,0.34765
min,-65.193000,-179.991000,0.000000,4.50000
25%,-30.747000,-73.901000,10.000000,4.60000
50%,-10.566000,-68.962000,30.000000,4.70000
75%,26.762000,105.271000,61.000000,5.00000
max,84.983700,179.940000,625.720000,7.30000


In [4]:
# ═══════════════════════════════════════════════════════════════
# PLOT TF-1: Dataset Overview — Annual/Monthly/Magnitude/Depth
# ═══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt, seaborn as sns, numpy as np
import matplotlib.gridspec as gridspec
import os; os.environ['MPLBACKEND'] = 'Agg'

fig = plt.figure(figsize=(20, 12))
fig.suptitle('Temporal Dataset Overview — Global Seismic Events', fontsize=16, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Annual counts (full width)
ax1 = fig.add_subplot(gs[0, :])
annual = df.set_index('time').resample('YE')['mag'].count()
c_map = plt.cm.viridis(np.linspace(0.2, 0.85, len(annual)))
bars = ax1.bar(annual.index.year, annual.values, color=c_map, edgecolor='white')
ax1.set_title('Annual Earthquake Count', fontsize=13, fontweight='bold')
ax1.set_xlabel('Year'); ax1.set_ylabel('Event Count'); ax1.grid(axis='y', alpha=0.3)
for b, v in zip(bars, annual.values):
    ax1.text(b.get_x()+b.get_width()/2, b.get_height()+20, str(int(v)),
             ha='center', va='bottom', fontsize=7, rotation=45)

# Magnitude distribution
ax2 = fig.add_subplot(gs[1, 0])
ax2.hist(df['mag'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax2.axvline(df['mag'].mean(), color='red', linestyle='--', lw=2,
            label=f'Mean={df["mag"].mean():.2f}')
ax2.set_title('Magnitude Distribution'); ax2.set_xlabel('Magnitude'); ax2.set_ylabel('Frequency')
ax2.legend(); ax2.grid(alpha=0.3)

# Depth distribution
ax3 = fig.add_subplot(gs[1, 1])
d_clip = df['depth'].clip(0, df['depth'].quantile(0.99))
ax3.hist(d_clip, bins=60, color='coral', edgecolor='white', alpha=0.85)
ax3.axvline(d_clip.median(), color='darkred', linestyle='--', lw=2,
            label=f'Median={d_clip.median():.1f} km')
ax3.set_title('Focal Depth Distribution'); ax3.set_xlabel('Depth (km)'); ax3.set_ylabel('Frequency')
ax3.legend(); ax3.grid(alpha=0.3)

# Monthly seasonality
ax4 = fig.add_subplot(gs[1, 2])
mc = df.groupby(df['time'].dt.month).size()
mnms = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
ax4.bar(mnms, [mc.get(m, 0) for m in range(1,13)],
        color=plt.cm.coolwarm(np.linspace(0.1,0.9,12)), edgecolor='white')
ax4.set_title('Events by Month (All Years)'); ax4.set_xlabel('Month'); ax4.set_ylabel('Count')
ax4.tick_params(axis='x', rotation=45); ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('plot_TF1_dataset_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot TF-1: Dataset Overview saved')

Plot TF-1: Dataset Overview saved


## 2b. Calendar Heatmap Visualization
A calendar heatmap shows **daily seismic activity** in a familiar month-grid layout, making seasonal and weekly patterns immediately visible.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

# Build daily event counts
_daily_cal = df.set_index('time').resample('D')['mag'].count().rename('count')
_daily_cal = _daily_cal.reset_index()
_daily_cal['year']    = _daily_cal['time'].dt.year
_daily_cal['month']   = _daily_cal['time'].dt.month
_daily_cal['day']     = _daily_cal['time'].dt.day
_daily_cal['weekday'] = _daily_cal['time'].dt.weekday   # Mon=0

# Pick 2 representative years
_years = sorted(_daily_cal['year'].unique())
_show_years = _years[-2:] if len(_years) >= 2 else _years

fig, axes = plt.subplots(len(_show_years), 1,
                         figsize=(22, 4 * len(_show_years)))
if len(_show_years) == 1:
    axes = [axes]

fig.patch.set_facecolor('white')                          
cmap = plt.cm.YlOrRd

for ax, yr in zip(axes, _show_years):
    ax.set_facecolor('white')                             
    _yr_data = _daily_cal[_daily_cal['year'] == yr].copy()

    _yr_data['week'] = _yr_data['time'].dt.isocalendar().week.astype(int)

    _vmax = _daily_cal['count'].quantile(0.97)
    _norm = mcolors.Normalize(vmin=0, vmax=_vmax)

    for _, row in _yr_data.iterrows():
        x = row['week'] - 1
        y = row['weekday']
        c = cmap(_norm(row['count']))
        rect = plt.Rectangle((x, y), 0.9, 0.9,
                              color=c, linewidth=0)
        ax.add_patch(rect)
        if row['count'] > 0:
            ax.text(x + 0.45, y + 0.45, int(row['count']),
                    ha='center', va='center', fontsize=4,
                    color='white' if _norm(row['count']) > 0.6 else '#111')  

    ax.set_xlim(-0.5, 53.5)
    ax.set_ylim(-0.5, 7.5)
    ax.set_yticks(range(7))
    ax.set_yticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'],
                       color='black', fontsize=9)          
    ax.set_xticks([])
    ax.set_title(f'Calendar Heatmap  –  Daily Earthquake Count  ({yr})',
                 color='black', fontsize=13, fontweight='bold', pad=10)  

    # Month labels
    for m in range(1, 13):
        _m_rows = _yr_data[_yr_data['month'] == m]
        if not _m_rows.empty:
            _wx = _m_rows['week'].min() - 1
            ax.text(_wx, 7.2, pd.Timestamp(yr, m, 1).strftime('%b'),
                    color='#333', fontsize=8)              

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=_norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation='vertical',
                        fraction=0.02, pad=0.01)
    cbar.set_label('Daily Event Count', color='black', fontsize=9)       
    cbar.ax.yaxis.set_tick_params(color='black')                         
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='black')              

plt.suptitle('📅  Calendar Visualization of Seismic Activity',
             color='black', fontsize=15, fontweight='bold', y=1.01)      
plt.tight_layout()
plt.savefig('plot_TFCAL_calendar_heatmap.png', dpi=150,
            bbox_inches='tight', facecolor='white')                      
plt.show()
print('✅  Calendar heatmap saved.')


✅  Calendar heatmap saved.


In [6]:
# Create grid_id if not present (1°×1° cells)
if 'grid_id' not in df.columns:
    df['lat_bin'] = (df['latitude'] // 1).astype(int)
    df['lon_bin'] = (df['longitude'] // 1).astype(int)
    df['grid_id'] = df['lat_bin'].astype(str) + '_' + df['lon_bin'].astype(str)

# Cyclical time encodings (already exist but re-derive cleanly)
df['hour']       = df['time'].dt.hour
df['month']      = df['time'].dt.month
df['year']       = df['time'].dt.year
df['day_of_year']= df['time'].dt.dayofyear
df['day_of_week']= df['time'].dt.dayofweek

df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin']  = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']  = np.cos(2 * np.pi * df['month'] / 12)
df['doy_sin']    = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['doy_cos']    = np.cos(2 * np.pi * df['day_of_year'] / 365)

print('✅ Core columns and cyclical encodings ready')
print(df[['time','hour','month','hour_sin','hour_cos','month_sin','month_cos']].head(3))

✅ Core columns and cyclical encodings ready
                       time  hour  month  hour_sin  hour_cos  month_sin  \
0 2000-01-01 11:22:57+00:00    11      1  0.258819 -0.965926        0.5   
1 2000-01-05 20:30:26+00:00    20      1 -0.866025  0.500000        0.5   
2 2000-01-06 10:42:25+00:00    10      1  0.500000 -0.866025        0.5   

   month_cos  
0   0.866025  
1   0.866025  
2   0.866025  


In [7]:
# ═══════════════════════════════════════════════════════════════
# PLOT TF-2: Cyclical Temporal Encodings
# ═══════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(20, 10))
fig.suptitle('Cyclical Temporal Encodings — Preserving Circular Time Structure',
             fontsize=14, fontweight='bold')

# Hour polar
ax1 = fig.add_subplot(2, 3, 1, polar=True)
hc = df.groupby(df['time'].dt.hour).size()
th = np.linspace(0, 2*np.pi, 24, endpoint=False)
ax1.bar(th, [hc.get(h, 0) for h in range(24)], width=2*np.pi/24, align='edge',
        color=plt.cm.twilight_shifted(np.linspace(0,1,24)), alpha=0.85)
ax1.set_title('Events by Hour of Day', pad=12)
ax1.set_xticks(th[::2]); ax1.set_xticklabels([str(h) for h in range(0,24,2)], fontsize=8)

# Month polar
ax2 = fig.add_subplot(2, 3, 2, polar=True)
moc = df.groupby(df['time'].dt.month).size()
th_m = np.linspace(0, 2*np.pi, 12, endpoint=False)
ax2.bar(th_m, [moc.get(m, 0) for m in range(1,13)], width=2*np.pi/12, align='edge',
        color=plt.cm.hsv(np.linspace(0,1,12)), alpha=0.85)
ax2.set_title('Events by Month', pad=12)
ax2.set_xticks(th_m)
ax2.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], fontsize=8)

# Day of week bars
ax3 = fig.add_subplot(2, 3, 3)
dowc = df.groupby(df['time'].dt.dayofweek).size()
ax3.bar(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], [dowc.get(d,0) for d in range(7)],
        color=plt.cm.Set2(np.linspace(0,1,7)))
ax3.set_title('Events by Day of Week'); ax3.set_xlabel('Day'); ax3.set_ylabel('Count')
ax3.grid(axis='y', alpha=0.3)

# Hour sin/cos scatter
ax4 = fig.add_subplot(2, 3, 4)
sc4 = ax4.scatter(df['hour_sin'][:3000], df['hour_cos'][:3000],
                  c=df['hour'][:3000], cmap='twilight', s=8, alpha=0.4)
plt.colorbar(sc4, ax=ax4, label='Hour')
ax4.set_title('Hour Cyclical Encoding'); ax4.set_xlabel('sin(2π·h/24)'); ax4.set_ylabel('cos(2π·h/24)')
ax4.set_aspect('equal'); ax4.grid(alpha=0.3)

# Month sin/cos scatter
ax5 = fig.add_subplot(2, 3, 5)
sc5 = ax5.scatter(df['month_sin'][:3000], df['month_cos'][:3000],
                  c=df['month'][:3000], cmap='hsv', s=8, alpha=0.4)
plt.colorbar(sc5, ax=ax5, label='Month')
ax5.set_title('Month Cyclical Encoding'); ax5.set_xlabel('sin(2π·m/12)'); ax5.set_ylabel('cos(2π·m/12)')
ax5.set_aspect('equal'); ax5.grid(alpha=0.3)

# Day of year trend
ax6 = fig.add_subplot(2, 3, 6)
doyc = df.groupby(df['time'].dt.dayofyear).size()
ax6.plot(doyc.index, doyc.values, color='teal', lw=0.8)
ax6.fill_between(doyc.index, doyc.values, alpha=0.2, color='teal')
ax6.set_title('Events by Day of Year'); ax6.set_xlabel('Day'); ax6.set_ylabel('Count'); ax6.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plot_TF2_cyclical_encodings.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot TF-2: Cyclical Encodings saved')

KeyboardInterrupt: 

## 3. Inter-Event Time (IET)

In [13]:
# ── Global IET ─────────────────────────────────────────────────────────────
df['inter_event_time_hrs'] = df['time'].diff().dt.total_seconds() / 3600
df['inter_event_time_hrs'] = df['inter_event_time_hrs'].fillna(0)

# Log-transform (IET is heavy-tailed)
df['log_iet'] = np.log1p(df['inter_event_time_hrs'])

# ── Per-grid-cell IET ──────────────────────────────────────────────────────
df = df.sort_values(['grid_id', 'time'])
df['iet_per_cell_hrs'] = df.groupby('grid_id')['time'].diff().dt.total_seconds() / 3600
df['iet_per_cell_hrs'] = df['iet_per_cell_hrs'].fillna(0)
df = df.sort_values('time').reset_index(drop=True)

print('Inter-event time statistics (hours):')
print(df['inter_event_time_hrs'].describe())

Inter-event time statistics (hours):
count    3153.000000
mean       70.618096
std       208.345647
min         0.000000
25%         9.138056
50%        28.544722
75%        68.825556
max      6897.003056
Name: inter_event_time_hrs, dtype: float64


In [14]:
# FIX: force safe backend (no GUI issues / broken backend imports)
import os
os.environ['MPLBACKEND'] = 'Agg'

import matplotlib.pyplot as plt

# ── Visualise IET distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Raw IET histogram (clipped)
iet_clipped = df['inter_event_time_hrs'].clip(
    upper=df['inter_event_time_hrs'].quantile(0.99)
)
axes[0].hist(iet_clipped, bins=60)
axes[0].set_title('IET Distribution (raw, clipped 99th pct)')
axes[0].set_xlabel('Inter-Event Time (hours)')
axes[0].set_ylabel('Count')

# Log-IET histogram
axes[1].hist(df['log_iet'], bins=60)
axes[1].set_title('Log(1 + IET) Distribution')
axes[1].set_xlabel('log(1 + IET)')
axes[1].set_ylabel('Count')

# Monthly event count (time-series)
monthly = df.set_index('time').resample('ME')['mag'].count()
axes[2].fill_between(monthly.index, monthly.values, alpha=0.6)
axes[2].set_title('Monthly Earthquake Count (global)')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Events / month')

plt.suptitle('Inter-Event Time Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('iet_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ IET plots saved')

✅ IET plots saved


## 4. Time Since Last Major Event (M ≥ 6)

In [15]:
# ── Major events (M ≥ 6) ──────────────────────────────────────────────────
major = df[df['mag'] >= 6.0][['time']].copy().rename(columns={'time': 'major_time'})
major = major.sort_values('major_time').reset_index(drop=True)

# For each event, find the most recent major event BEFORE it
def time_since_major(event_time, major_times):
    past = major_times[major_times < event_time]
    if len(past) == 0:
        return np.nan
    return (event_time - past.iloc[-1]).total_seconds() / 3600

# Vectorised using merge_asof
df_tmp = df[['time']].copy()
df_tmp = df_tmp.sort_values('time')
major_sorted = major.sort_values('major_time')

merged = pd.merge_asof(
    df_tmp,
    major_sorted,
    left_on='time',
    right_on='major_time',
    direction='backward'
)
merged['time_since_major_hrs'] = (merged['time'] - merged['major_time']).dt.total_seconds() / 3600
merged = merged.sort_values('time').reset_index(drop=True)

df['time_since_major_hrs'] = merged['time_since_major_hrs'].values
df['time_since_major_hrs'] = df['time_since_major_hrs'].fillna(df['time_since_major_hrs'].median())

print('time_since_major_hrs stats:')
print(df['time_since_major_hrs'].describe())

time_since_major_hrs stats:
count     3153.000000
mean      3424.777594
std       4234.886932
min          0.000000
25%        879.141944
50%       2281.283611
75%       4569.721667
max      49722.843611
Name: time_since_major_hrs, dtype: float64


In [16]:
# Aftershock flag: occurred within 30 days (720 hrs) of M≥6 event
df['is_aftershock'] = (df['time_since_major_hrs'] <= 720).astype(int)
aftershock_rate = df['is_aftershock'].mean() * 100
print(f'Aftershock flag: {df["is_aftershock"].sum():,} events ({aftershock_rate:.1f}% of total)')

Aftershock flag: 690 events (21.9% of total)


In [17]:
# ═══════════════════════════════════════════════════════════════
# PLOT TF-3: Aftershock Classification Timeline
# ═══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 1, figsize=(18, 9))
fig.suptitle('Aftershock Classification & Time-Since-Major-Event Analysis',
             fontsize=14, fontweight='bold')

ax = axes[0]
normal   = df[df['is_aftershock'] == 0]
ashock   = df[df['is_aftershock'] == 1]
smpl_n   = normal.sample(min(5000, len(normal)), random_state=42)
ax.scatter(smpl_n['time'],  smpl_n['mag'],  s=5, alpha=0.25, color='steelblue', label='Background')
ax.scatter(ashock['time'],  ashock['mag'],  s=8, alpha=0.5,  color='crimson',   label=f'Aftershocks (n={len(ashock):,})')
maj = df[df['mag'] >= 6.0]
ax.scatter(maj['time'], maj['mag'], s=80, marker='*', color='gold',
           edgecolors='black', lw=0.5, zorder=5, label=f'M≥6 (n={len(maj):,})')
ax.set_title('Event Timeline — Aftershock Classification')
ax.set_xlabel('Date'); ax.set_ylabel('Magnitude')
ax.legend(loc='upper left'); ax.grid(alpha=0.2)

ax2 = axes[1]
tsm_d = df['time_since_major_hrs'].clip(0, df['time_since_major_hrs'].quantile(0.98)) / 24
ax2.hist(tsm_d, bins=80, color='mediumpurple', edgecolor='white', alpha=0.8,
         density=True, label='PDF')
ax2.axvline(30, color='red', linestyle='--', lw=2, label='30-day threshold')
ax2r = ax2.twinx()
sv = np.sort(tsm_d.dropna()); cdf = np.arange(1,len(sv)+1)/len(sv)
ax2r.plot(sv, cdf, color='darkorange', lw=2, label='CDF')
ax2r.set_ylabel('CDF', color='darkorange'); ax2r.tick_params(axis='y', labelcolor='darkorange')
ax2.set_title('Time Since Last M≥6 Event — Distribution')
ax2.set_xlabel('Days Since Major Event'); ax2.set_ylabel('Probability Density')
h1, l1 = ax2.get_legend_handles_labels(); h2, l2 = ax2r.get_legend_handles_labels()
ax2.legend(h1+h2, l1+l2, loc='upper right'); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plot_TF3_aftershock_timeline.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot TF-3: Aftershock Timeline saved')

Plot TF-3: Aftershock Timeline saved


## 5. Rolling Statistics Windows

In [19]:
# ── Set time index for rolling computations ────────────────────────────────
df_ts = df.set_index('time').sort_index()

# Rolling count of events in last 7 / 30 / 90 days
# Use a rolling window on a dummy series of 1s
ones = pd.Series(1, index=df_ts.index)
df_ts['rolling_count_7d']   = ones.rolling('7D',  min_periods=1).sum()
df_ts['rolling_count_30d']  = ones.rolling('30D', min_periods=1).sum()
df_ts['rolling_count_90d']  = ones.rolling('90D', min_periods=1).sum()

# Rolling mean magnitude
df_ts['rolling_mag_7d']  = df_ts['mag'].rolling('7D',  min_periods=1).mean()
df_ts['rolling_mag_30d'] = df_ts['mag'].rolling('30D', min_periods=1).mean()
df_ts['rolling_mag_90d'] = df_ts['mag'].rolling('90D', min_periods=1).mean()

# Rolling max magnitude
df_ts['rolling_max_mag_30d'] = df_ts['mag'].rolling('30D', min_periods=1).max()

# Rolling std magnitude
df_ts['rolling_std_mag_30d'] = df_ts['mag'].rolling('30D', min_periods=2).std().fillna(0)

print('✅ Rolling features computed')
print(df_ts[['rolling_count_7d','rolling_count_30d','rolling_count_90d',
             'rolling_mag_7d','rolling_mag_30d']].describe())

✅ Rolling features computed
       rolling_count_7d  rolling_count_30d  rolling_count_90d  rolling_mag_7d  \
count       3153.000000        3153.000000        3153.000000     3153.000000   
mean           5.503330          19.053283          53.800190        4.813722   
std            3.640064          10.421069          26.678418        0.204675   
min            1.000000           1.000000           1.000000        4.500000   
25%            3.000000          11.000000          33.000000        4.685714   
50%            5.000000          18.000000          55.000000        4.790000   
75%            7.000000          26.000000          74.000000        4.900000   
max           26.000000          51.000000         127.000000        6.600000   

       rolling_mag_30d  
count      3153.000000  
mean          4.811935  
std           0.124304  
min           4.500000  
25%           4.732258  
50%           4.800000  
75%           4.873913  
max           5.950000  


In [20]:
# ── Visualise rolling counts ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Downsample to daily for plotting
daily = df_ts.resample('D').agg(
    count=('mag','count'),
    rc_7d=('rolling_count_7d','last'),
    rc_30d=('rolling_count_30d','last'),
    mean_mag=('mag','mean')
).dropna()

axes[0].plot(daily.index, daily['rc_7d'],  label='7-day rolling count',  alpha=0.8, linewidth=1.0)
axes[0].plot(daily.index, daily['rc_30d'], label='30-day rolling count', alpha=0.8, linewidth=1.2, color='orange')
axes[0].set_title('Rolling Event Counts Over Time')
axes[0].set_ylabel('# Events')
axes[0].legend()

axes[1].plot(daily.index, daily['mean_mag'], color='crimson', linewidth=0.8, alpha=0.7, label='Daily mean mag')
axes[1].axhline(y=6.0, color='darkred', linestyle='--', linewidth=0.8, label='M=6 threshold')
axes[1].set_title('Daily Mean Magnitude Over Time')
axes[1].set_ylabel('Magnitude')
axes[1].legend()

plt.tight_layout()
plt.savefig('rolling_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 5b. Line + Bar Chart (Juxtaposed) – Simple Time Series
Juxtaposing a **bar chart** (daily counts) with a **line chart** (rolling average) on a shared time axis is one of the most fundamental and widely-used time-series visualizations.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# PLOT TF-LB: Line + Bar Chart (Juxtaposed) — Simple Time Series
# ════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

_daily_lb = df.set_index('time').resample('D').agg(
    count=('mag', 'count'),
    mean_mag=('mag', 'mean')
).fillna(0)

# Keep last 3 years for readability
_daily_lb = _daily_lb[_daily_lb.index.year >= _daily_lb.index.year.max() - 2]
_daily_lb['roll7']  = _daily_lb['count'].rolling(7,  min_periods=1).mean()
_daily_lb['roll30'] = _daily_lb['count'].rolling(30, min_periods=1).mean()

fig, ax1 = plt.subplots(figsize=(20, 6))
fig.patch.set_facecolor('white')                          
ax1.set_facecolor('white')                                

# ── BAR: daily count ──
ax1.bar(_daily_lb.index, _daily_lb['count'],
        color='#3a6bc4', alpha=0.55, width=1.0, label='Daily Count (bar)')

# ── LINE: rolling averages (shared axis) ──
ax1.plot(_daily_lb.index, _daily_lb['roll7'],
         color='#d4820a', linewidth=1.6, label='7-day Rolling Avg')    # ← darkened orange
ax1.plot(_daily_lb.index, _daily_lb['roll30'],
         color='#c0392b', linewidth=2.0, linestyle='--', label='30-day Rolling Avg')  # ← darkened red

ax1.set_ylabel('Event Count', color='black', fontsize=11)              
ax1.tick_params(axis='y', colors='black')                              
ax1.tick_params(axis='x', colors='#333', rotation=30)                 
for spine in ax1.spines.values():
    spine.set_edgecolor('#bbb')                                        
ax1.yaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=6))

# ── SECOND y-axis: mean magnitude as line ──
ax2 = ax1.twinx()
ax2.set_facecolor('white')                                             
ax2.plot(_daily_lb.index,
         _daily_lb['mean_mag'].rolling(14, min_periods=1).mean(),
         color='#27ae60', linewidth=1.3, alpha=0.9, label='14-day Mean Mag')  # ← darkened green
ax2.set_ylabel('Mean Magnitude', color='#27ae60', fontsize=10)        
ax2.tick_params(axis='y', colors='#27ae60')                           
for spine in ax2.spines.values():
    spine.set_edgecolor('#bbb')                                        

# Legend
lines1, lab1 = ax1.get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lab1 + lab2,
           facecolor='white', edgecolor='#bbb',                        
           labelcolor='black', fontsize=9, loc='upper left')           

ax1.set_title('📊  Line + Bar Chart (Juxtaposed) — Daily Seismic Events & Rolling Averages',
              color='black', fontsize=13, fontweight='bold', pad=12)   
ax1.set_xlabel('Date', color='#333', fontsize=10)                      

plt.tight_layout()
plt.savefig('plot_TFLB_line_bar_juxtaposed.png', dpi=150,
            bbox_inches='tight', facecolor='white')                    
plt.show()
print('✅  Line+Bar juxtaposed chart saved.')


✅  Line+Bar juxtaposed chart saved.


## 6. Gutenberg-Richter Rolling b-Value

In [23]:
def compute_b_value(magnitudes, mc=4.5):
    """Maximum Likelihood Estimate of Gutenberg-Richter b-value."""
    mags = magnitudes[magnitudes >= mc]
    if len(mags) < 10:
        return np.nan
    return np.log10(np.e) / (mags.mean() - mc)

# Rolling 90-day b-value — computed on monthly-resampled data for efficiency
monthly_mags = df_ts['mag'].resample('ME').apply(list)

b_values = []
dates = []
window = 3   # 3 months = ~90 days
for i in range(window - 1, len(monthly_mags)):
    combined = []
    for j in range(i - window + 1, i + 1):
        combined.extend(monthly_mags.iloc[j])
    b = compute_b_value(np.array(combined))
    b_values.append(b)
    dates.append(monthly_mags.index[i])

b_df = pd.DataFrame({'date': dates, 'b_value_rolling': b_values}).set_index('date')

# Merge back to main dataframe
df_ts['b_value_rolling'] = np.nan
# Forward-fill b-value for each event in that month
df_ts = df_ts.sort_index()
b_df = b_df.sort_index()
df_ts['b_value_rolling'] = pd.merge_asof(
    df_ts[['mag']].reset_index(),
    b_df.reset_index().rename(columns={'date':'time'}),
    on='time',
    direction='backward'
)['b_value_rolling'].values
df_ts['b_value_rolling'] = df_ts['b_value_rolling'].ffill().bfill()

print(f'✅ Rolling b-value computed: mean={b_df["b_value_rolling"].mean():.3f}')
print(b_df.describe())

✅ Rolling b-value computed: mean=1.476
       b_value_rolling
count       225.000000
mean          1.475757
std           0.393679
min           0.608012
25%           1.191662
50%           1.447648
75%           1.698652
max           2.961099


In [24]:
# ── Gutenberg-Richter law fit ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# G-R magnitude-frequency plot
mag_bins = np.arange(df['mag'].min(), df['mag'].max() + 0.1, 0.1)
mag_counts = []
for m in mag_bins:
    mag_counts.append((df['mag'] >= m).sum())

log_counts = np.log10(np.array(mag_counts) + 1)
axes[0].plot(mag_bins, log_counts, 'o', color='steelblue', markersize=3, label='Observed')

# Fit linear region (M 4.5–7.5)
mask = (mag_bins >= 4.5) & (mag_bins <= 7.5)
from numpy.polynomial.polynomial import polyfit as npfit
coeffs = np.polyfit(mag_bins[mask], log_counts[mask], 1)
fit_line = np.polyval(coeffs, mag_bins[mask])
b_fit = -coeffs[0]
axes[0].plot(mag_bins[mask], fit_line, 'r--', linewidth=2,
             label=f'G-R fit: b={b_fit:.2f}')
axes[0].set_title('Gutenberg-Richter Relationship')
axes[0].set_xlabel('Magnitude (M)')
axes[0].set_ylabel('log₁₀(N ≥ M)')
axes[0].legend()

# Rolling b-value over time
axes[1].plot(b_df.index, b_df['b_value_rolling'], color='darkgreen', linewidth=1.5)
axes[1].axhline(y=1.0, color='gray', linestyle='--', linewidth=0.8, label='b=1.0 reference')
axes[1].fill_between(b_df.index, b_df['b_value_rolling'], alpha=0.2, color='darkgreen')
axes[1].set_title('Rolling 90-day Gutenberg-Richter b-value')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('b-value')
axes[1].legend()

plt.tight_layout()
plt.savefig('bvalue_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Global b-value fit: {b_fit:.3f}')

✅ Global b-value fit: 1.124


## 7. Assemble Final Temporal Feature Matrix

In [26]:
# Merge rolling features back from df_ts
rolling_cols = ['rolling_count_7d','rolling_count_30d','rolling_count_90d',
                'rolling_mag_7d','rolling_mag_30d','rolling_mag_90d',
                'rolling_max_mag_30d','rolling_std_mag_30d','b_value_rolling']

df_ts_reset = df_ts[rolling_cols].reset_index()  # time as column

# Re-index df on time for merge
df_sorted = df.sort_values('time').reset_index(drop=True)

df_merged = pd.merge_asof(
    df_sorted,
    df_ts_reset.sort_values('time'),
    on='time',
    direction='backward'
)

# Final temporal feature matrix
temporal_feat_cols = [
    'time', 'latitude', 'longitude', 'depth', 'mag',
    # Raw time
    'year', 'month', 'day_of_year', 'hour', 'day_of_week',
    # Cyclical encodings
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
    # IET
    'inter_event_time_hrs', 'log_iet', 'iet_per_cell_hrs',
    # Time since major
    'time_since_major_hrs', 'is_aftershock',
    # Rolling
    'rolling_count_7d', 'rolling_count_30d', 'rolling_count_90d',
    'rolling_mag_7d', 'rolling_mag_30d', 'rolling_mag_90d',
    'rolling_max_mag_30d', 'rolling_std_mag_30d',
    # b-value & Omori
    'b_value_rolling', 'omori_decay_rate',
    # Grid
    'grid_id'
]

temporal_feat_cols = [c for c in temporal_feat_cols if c in df_merged.columns]
temporal_df = df_merged[temporal_feat_cols].copy()
temporal_df = temporal_df.fillna(temporal_df.median(numeric_only=True))

print(f'✅ Temporal feature matrix shape: {temporal_df.shape}')
temporal_df.describe()

✅ Temporal feature matrix shape: (3153, 31)


,latitude,longitude,depth,mag,year,month,day_of_year,hour,day_of_week,hour_sin,...,is_aftershock,rolling_count_7d,rolling_count_30d,rolling_count_90d,rolling_mag_7d,rolling_mag_30d,rolling_mag_90d,rolling_max_mag_30d,rolling_std_mag_30d,b_value_rolling
count,3153.000000,3153.000000,3153.000000,3153.00000,3153.000000,3153.000000,3153.000000,3153.000000,3153.000000,3.153000e+03,...,3153.000000,3153.000000,3153.000000,3153.000000,3153.000000,3153.000000,3153.000000,3153.000000,3153.000000,3153.000000
mean,-2.925395,-8.441238,54.971366,4.81125,2010.454171,6.385347,178.782112,11.330479,3.017444,1.682133e-02,...,0.218839,5.503330,19.053283,53.800190,4.813722,4.811935,4.813575,5.624104,0.314354,1.480155
std,31.601950,109.342550,83.874967,0.34765,4.533702,3.423975,104.411773,7.019002,1.994284,7.015285e-01,...,0.413525,3.640064,10.421069,26.678418,0.204675,0.124304,0.089005,0.512839,0.131585,0.357471
min,-65.193000,-179.991000,0.000000,4.50000,2000.000000,1.000000,1.000000,0.000000,0.000000,-1.000000e+00,...,0.000000,1.000000,1.000000,1.000000,4.500000,4.500000,4.500000,4.500000,0.000000,0.608012
25%,-30.747000,-73.901000,10.000000,4.60000,2007.000000,3.000000,84.000000,5.000000,1.000000,-7.071068e-01,...,0.000000,3.000000,11.000000,33.000000,4.685714,4.732258,4.754167,5.300000,0.230940,1.211112
50%,-10.566000,-68.962000,30.000000,4.70000,2011.000000,6.000000,179.000000,11.000000,3.000000,1.224647e-16,...,0.000000,5.000000,18.000000,55.000000,4.790000,4.800000,4.806061,5.600000,0.299358,1.447648
75%,26.762000,105.271000,61.000000,5.00000,2014.000000,9.000000,267.000000,18.000000,5.000000,7.071068e-01,...,0.000000,7.000000,26.000000,74.000000,4.900000,4.873913,4.863714,5.900000,0.376516,1.698652
max,84.983700,179.940000,625.720000,7.30000,2025.000000,12.000000,366.000000,23.000000,6.000000,1.000000e+00,...,1.000000,26.000000,51.000000,127.000000,6.600000,5.950000,6.033333,7.300000,1.217580,2.961099


In [27]:
# ═══════════════════════════════════════════════════════════════
# PLOT TF-5: Temporal Feature Distributions
# ═══════════════════════════════════════════════════════════════
feat_plot = ['inter_event_time_hrs','log_iet','time_since_major_hrs',
             'rolling_count_7d','rolling_count_30d','rolling_mag_7d',
             'b_value_rolling','omori_decay_rate']
feat_plot = [c for c in feat_plot if c in temporal_df.columns]
n = len(feat_plot); ncols = 4; nrows = (n+ncols-1)//ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 4*nrows))
fig.suptitle('All Temporal Feature Distributions (1st–99th percentile clipped)',
             fontsize=14, fontweight='bold')
axes = axes.flatten()
pal = plt.cm.tab10(np.linspace(0,1,n))

for i, (col, clr) in enumerate(zip(feat_plot, pal)):
    ax = axes[i]
    v = temporal_df[col].dropna()
    vc = v.clip(v.quantile(0.01), v.quantile(0.99))
    ax.hist(vc, bins=50, color=clr, edgecolor='white', alpha=0.85)
    ax.axvline(vc.median(), color='black', linestyle='--', lw=1.2,
               label=f'Med={vc.median():.2f}')
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_xlabel('Value'); ax.set_ylabel('Count'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

for j in range(n, len(axes)): axes[j].set_visible(False)
plt.tight_layout()
plt.savefig('plot_TF5_feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot TF-5: Feature Distributions saved')

Plot TF-5: Feature Distributions saved


In [ ]:
# ── Feature correlation heatmap ────────────────────────────────────────────
numeric_feats = temporal_df.select_dtypes(include=np.number).drop(
    columns=['latitude','longitude','year'], errors='ignore'
)

corr = numeric_feats.corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # upper triangle
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, square=True, linewidths=0.5,
    annot_kws={'size': 7}, ax=ax, vmin=-1, vmax=1
)
ax.set_title('Temporal Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('temporal_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save temporal feature matrix ───────────────────────────────────────────
temporal_df.to_csv('temporal_features.csv', index=False)
print(f'✅ Saved temporal_features.csv  ({temporal_df.shape[0]:,} rows × {temporal_df.shape[1]} cols)')
print('\nFinal feature list:')
for i, c in enumerate(temporal_df.columns, 1):
    print(f'  {i:2d}. {c}')

✅ Saved temporal_features.csv  (3,153 rows × 32 cols)

Final feature list:
   1. time
   2. latitude
   3. longitude
   4. depth
   5. mag
   6. year
   7. month
   8. day_of_year
   9. hour
  10. day_of_week
  11. hour_sin
  12. hour_cos
  13. month_sin
  14. month_cos
  15. doy_sin
  16. doy_cos
  17. inter_event_time_hrs
  18. log_iet
  19. iet_per_cell_hrs
  20. time_since_major_hrs
  21. is_aftershock
  22. rolling_count_7d
  23. rolling_count_30d
  24. rolling_count_90d
  25. rolling_mag_7d
  26. rolling_mag_30d
  27. rolling_mag_90d
  28. rolling_max_mag_30d
  29. rolling_std_mag_30d
  30. b_value_rolling
  31. omori_decay_rate
  32. grid_id


## 9. Summary

| Feature Group | Features | Description |
|---|---|---|
| **Raw time** | year, month, day_of_year, hour, day_of_week | Basic temporal identifiers |
| **Cyclical encoding** | hour_sin/cos, month_sin/cos, doy_sin/cos | Preserves circular time structure |
| **Inter-event time** | IET (hrs), log-IET, per-cell IET | Temporal spacing between events |
| **Post-major event** | time_since_major_hrs, is_aftershock | Proximity to M≥6 events |
| **Rolling statistics** | count 7/30/90d, mean/max/std mag | Seismicity rate & magnitude trends |
| **Gutenberg-Richter** | b_value_rolling (90-day) | Seismicity regime indicator |
| **Omori–Utsu** | omori_decay_rate | Aftershock rate via physical model |

**Output:** `temporal_features.csv` — used as input to `temporal_ml_models.ipynb` and `temporal_mining.ipynb`
